In [1]:
import os

os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

try:
    from google.protobuf import message_factory

    if not hasattr(message_factory.MessageFactory, "GetPrototype"):
        message_factory.MessageFactory.GetPrototype = (
            lambda self, descriptor: self.GetMessageClass(descriptor)
        )
except Exception:
    pass

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing import image as kimage
from tensorflow.keras.models import load_model, Sequential
from tensorflow.keras.layers import GlobalAveragePooling2D, Dropout, Dense, Input
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical

tf.random.set_seed(42)
np.random.seed(42)




2026-05-02 18:19:34.588121: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-02 18:19:34.601188: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777745974.616119      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777745974.620615      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-02 18:19:34.637088: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [2]:
WORK_DIR = "../input/cassava-leaf-disease-classification/"




In [3]:
class SigmoidFocalCrossEntropy(tf.keras.losses.Loss):
    def __init__(self, alpha=0.25, gamma=2.0, from_logits=False, **kwargs):
        super().__init__(**kwargs)
        self.alpha = alpha
        self.gamma = gamma
        self.from_logits = from_logits

    def call(self, y_true, y_pred):
        if self.from_logits:
            y_pred = tf.sigmoid(y_pred)
        y_pred = tf.clip_by_value(
            y_pred, tf.keras.backend.epsilon(), 1 - tf.keras.backend.epsilon()
        )
        cross_entropy = -y_true * tf.math.log(y_pred) - (1 - y_true) * tf.math.log(
            1 - y_pred
        )
        weight = self.alpha * y_true + (1 - self.alpha) * (1 - y_true)
        focal_loss = weight * ((1 - y_pred) ** self.gamma) * cross_entropy
        return tf.reduce_sum(focal_loss, axis=-1)




In [4]:
model_path = "/kaggle/input/finalsubmit/myfinal_model555.h5"
custom_objects = {"SigmoidFocalCrossEntropy": SigmoidFocalCrossEntropy}
use_feature_extractor = (
    False  # indicates whether we need to extract features for inference
)

if os.path.exists(model_path):
    model = load_model(model_path, custom_objects=custom_objects)
else:
    train_df = pd.read_csv(os.path.join(WORK_DIR, "train.csv"))
    train_df["label"] = train_df["label"].astype(str)

    base_extractor = EfficientNetB0(
        weights="imagenet",
        include_top=False,
        input_shape=(224, 224, 3),
        pooling="avg",
    )
    base_extractor.trainable = False

    BATCH_SIZE_FEAT = 256
    train_gen_feat = tf.keras.preprocessing.image.ImageDataGenerator(
        preprocessing_function=preprocess_input
    ).flow_from_dataframe(
        dataframe=train_df,
        directory=os.path.join(WORK_DIR, "train_images"),
        x_col="image_id",
        y_col="label",
        target_size=(224, 224),
        batch_size=BATCH_SIZE_FEAT,
        class_mode=None,
        shuffle=False,
        workers=4,
        use_multiprocessing=True,
    )

    steps_feat = int(np.ceil(train_gen_feat.samples / BATCH_SIZE_FEAT))
    train_features = base_extractor.predict(
        train_gen_feat, steps=steps_feat, verbose=1
    )  # (num_samples, 1280)

    y_int = train_df["label"].astype(int).values
    train_labels = to_categorical(y_int, num_classes=5)

    # ----- Modified classifier: larger dense layer and higher dropout -----
    classifier = Sequential(
        [
            Dropout(0.3, input_shape=(train_features.shape[1],)),
            Dense(512, activation="relu"),
            Dropout(0.3),
            Dense(5, activation="softmax"),
        ]
    )
    # ---------------------------------------------------------------------

    classifier.compile(
        optimizer=tf.keras.optimizers.Adam(),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )

    # ----- Longer early‑stopping patience to allow more epochs -----
    es = EarlyStopping(patience=10, restore_best_weights=True, monitor="val_accuracy")
    # -----------------------------------------------------------------
    rl = ReduceLROnPlateau(patience=3, factor=0.5, monitor="val_accuracy")

    classifier.fit(
        train_features,
        train_labels,
        epochs=60,  # increased from 30 to give the larger model time to learn
        batch_size=64,
        validation_split=0.1,
        callbacks=[es, rl],
        verbose=2,
    )

    model = classifier
    use_feature_extractor = True




2026-05-02 18:19:44.480142: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Found 18721 validated image filenames.


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


74/74 ━━━━━━━━━━━━━━━━━━━━ 241s 3s/step
Epoch 1/60


/usr/local/lib/python3.11/dist-packages/keras/src/layers/regularization/dropout.py:42: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


264/264 - 2s - 8ms/step - accuracy: 0.7144 - loss: 0.7784 - val_accuracy: 0.7453 - val_loss: 0.7018 - learning_rate: 0.0010
Epoch 2/60
264/264 - 1s - 4ms/step - accuracy: 0.7481 - loss: 0.6834 - val_accuracy: 0.7560 - val_loss: 0.6724 - learning_rate: 0.0010
Epoch 3/60
264/264 - 1s - 5ms/step - accuracy: 0.7657 - loss: 0.6480 - val_accuracy: 0.7656 - val_loss: 0.6641 - learning_rate: 0.0010
Epoch 4/60
264/264 - 1s - 4ms/step - accuracy: 0.7702 - loss: 0.6274 - val_accuracy: 0.7587 - val_loss: 0.6691 - learning_rate: 0.0010
Epoch 5/60
264/264 - 1s - 5ms/step - accuracy: 0.7766 - loss: 0.6051 - val_accuracy: 0.7635 - val_loss: 0.6579 - learning_rate: 0.0010
Epoch 6/60
264/264 - 1s - 4ms/step - accuracy: 0.7873 - loss: 0.5868 - val_accuracy: 0.7624 - val_loss: 0.6662 - learning_rate: 0.0010
Epoch 7/60
264/264 - 1s - 4ms/step - accuracy: 0.8025 - loss: 0.5433 - val_accuracy: 0.7704 - val_loss: 0.6628 - learning_rate: 5.0000e-04
Epoch 8/60
264/264 - 1s - 4ms/step - accuracy: 0.8096 - loss: 

In [5]:
test_dir = os.path.join(WORK_DIR, "test_images")
image_names = sorted(
    [
        f
        for f in os.listdir(test_dir)
        if os.path.isfile(os.path.join(test_dir, f))
        and f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]
)

batch_size = 32
pred_labels = []

for i in range(0, len(image_names), batch_size):
    batch_names = image_names[i : i + batch_size]
    batch_images = np.empty((len(batch_names), 224, 224, 3), dtype=np.float32)
    for idx, name in enumerate(batch_names):
        img_path = os.path.join(test_dir, name)
        img = kimage.load_img(img_path, target_size=(224, 224))
        batch_images[idx] = kimage.img_to_array(img)
    x_batch = preprocess_input(batch_images)

    if use_feature_extractor:
        feats = base_extractor.predict(x_batch, verbose=0)
        preds = model.predict(feats, verbose=0)
    else:
        preds = model.predict(x_batch, verbose=0)

    batch_preds = np.argmax(preds, axis=-1).astype(int)
    pred_labels.extend(batch_preds.tolist())

submission = pd.DataFrame({"image_id": image_names, "label": pred_labels})
submission.to_csv("submission.csv", index=False)